<a href="https://colab.research.google.com/github/chantakorn-uranukul-5792092/chantakorn-uranukul-5792092-AnalyticsEngineering/blob/main/5_02_practice_weekly_ops_attempt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

This is set to be the same time given in the real test; the same style of notebook and problem; and as close to the real thing as possible. However, of course, the problem is different; the data is different; and you will do it asychronously (which is also different).

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

In the real test you upload the zip to the Class Test area on my.wbs. Here, keep it so you can compare your work with the model answer afterwards.

**Started at:** ____  **Stopped at:** ____  **Furthest step I reached (see the brief):** ____

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 14,494 rows, 5 columns
    ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'delivery_fee']
order_status_events: 56,812 rows, 4 columns
    ['event_id', 'order_id', 'status', 'event_ts']
order_lines: 36,120 rows, 5 columns
    ['order_id', 'line_no', 'item_name', 'quantity', 'unit_price']
payments: 16,423 rows, 5 columns
    ['payment_id', 'order_id', 'payment_type', 'amount', 'paid_at']
restaurants: 67 rows, 5 columns
    ['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'effective_from']
customers: 4,200 rows, 3 columns
    ['customer_id', 'signup_date', 'home_city']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


,n_rows,n_distinct_orders
0,14494,14494


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


(14494, 5)
order_id          object
customer_id       object
restaurant_id     object
order_ts          object
delivery_fee     float64
dtype: object


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


4073 different customer_id values in orders


## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

Let start by looking into each table and understand all the data we have

Prompt : showing 5 rows from all the table

In [5]:
from IPython.display import display

for name, df in tables.items():
    print(f"=== Table: {name} ({df.shape[0]:,} rows) ===")
    display(df.head(5))
    print("\n")

=== Table: orders (14,494 rows) ===


,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00




=== Table: order_status_events (56,812 rows) ===


,event_id,order_id,status,event_ts
0,E027809,O007104,accepted,2026-04-29T13:28:57Z
1,E018209,O004658,placed,2026-04-08T19:11:30Z
2,E043792,O011178,accepted,2026-06-02T17:07:47Z
3,E021428,O005468,delivered,2026-04-15T13:43:36Z
4,E037134,O009483,picked_up,2026-05-19T12:18:18Z




=== Table: order_lines (36,120 rows) ===


,order_id,line_no,item_name,quantity,unit_price
0,O000001,1,Ramen,3,12.0
1,O000002,1,Soft drink,1,2.0
2,O000002,2,Side salad,1,3.5
3,O000002,3,Burrito,3,8.0
4,O000003,1,Fries,1,3.0


... (2 outputs left out)



=== Table: restaurants (67 rows) ===


,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R064,Fresh Vegan 64,Vegan,London,2025-01-01
1,R012,Royal Burgers 12,Burgers,Bristol,2025-01-01
2,R028,Grand Burgers 28,Burgers,Birmingham,2025-01-01
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
4,R026,Corner Indian 26,Indian,London,2025-01-01




=== Table: customers (4,200 rows) ===


,customer_id,signup_date,home_city
0,C00001,2024-10-15,London
1,C00002,2025-05-17,Leeds
2,C00003,2026-01-25,Leeds
3,C00004,2024-09-14,London
4,C00005,2024-01-25,Manchester


okay so we seems to have 6 table that we have to clean. I will first checking for missing value from all these tables

Prompt : checking for any missing value from the 6 tables. I want to see the output by table and by column

In [6]:
for name, df in tables.items():
    print(f"=== Missing Values in '{name}' ===")
    missing_counts = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df)) * 100
    missing_df = pd.DataFrame({
        'Missing Count': missing_counts,
        'Percentage (%)': missing_pct
    })
    display(missing_df)
    print("\n")

=== Missing Values in 'orders' ===


,Missing Count,Percentage (%)
order_id,0,0.0
customer_id,0,0.0
restaurant_id,0,0.0
order_ts,0,0.0
delivery_fee,0,0.0




=== Missing Values in 'order_status_events' ===


,Missing Count,Percentage (%)
event_id,0,0.0
order_id,0,0.0
status,0,0.0
event_ts,0,0.0




=== Missing Values in 'order_lines' ===


,Missing Count,Percentage (%)
order_id,0,0.0
line_no,0,0.0
item_name,0,0.0
quantity,0,0.0
unit_price,0,0.0


... (2 outputs left out)



=== Missing Values in 'restaurants' ===


,Missing Count,Percentage (%)
restaurant_id,0,0.0
restaurant_name,0,0.0
cuisine,0,0.0
city,0,0.0
effective_from,0,0.0




=== Missing Values in 'customers' ===


,Missing Count,Percentage (%)
customer_id,0,0.0
signup_date,0,0.0
home_city,0,0.0


Okay nice. There seems to be no missing value across 6 tables so I will move on to check for any exact duplicates in the 6 table

Prompt: check for the exact duplicates in the 6 tables. expect the output to be by table and also show the duplicate found

In [7]:
for name, df in tables.items():
    duplicate_count = df.duplicated().sum()
    print(f"=== Exact Duplicates in '{name}': {duplicate_count} ===")
    if duplicate_count > 0:
        display(df[df.duplicated(keep=False)].head(5))
    print("\n")

=== Exact Duplicates in 'orders': 0 ===


=== Exact Duplicates in 'order_status_events': 0 ===


=== Exact Duplicates in 'order_lines': 0 ===


=== Exact Duplicates in 'payments': 30 ===


,payment_id,order_id,payment_type,amount,paid_at
13,P012637,O011166,charge,70.99,2026-06-02 12:59:12
172,P016187,O014307,charge,42.49,2026-06-29 12:07:12
870,P013877,O012271,charge,19.49,2026-06-11 21:38:21
975,P001859,O001631,charge,38.49,2026-03-14 22:43:50
1673,P000927,O000815,charge,23.99,2026-03-08 11:53:53




=== Exact Duplicates in 'restaurants': 0 ===


=== Exact Duplicates in 'customers': 0 ===




Okay there is only duplicates in the table payments. Let investigate it more.

Prompt : filter out all the duplicate rows from table payments

In [8]:
payments_df = tables['payments']
duplicates = payments_df[payments_df.duplicated(keep=False)].sort_values(by='payment_id')
display(duplicates)

,payment_id,order_id,payment_type,amount,paid_at
5949,P000219,O000185,charge,25.49,2026-03-02 19:05:43
13373,P000219,O000185,charge,25.49,2026-03-02 19:05:43
1673,P000927,O000815,charge,23.99,2026-03-08 11:53:53
4665,P000927,O000815,charge,23.99,2026-03-08 11:53:53
6582,P000998,O000874,charge,50.99,2026-03-08 19:19:26
3747,P000998,O000874,charge,50.99,2026-03-08 19:19:26
975,P001859,O001631,charge,38.49,2026-03-14 22:43:50
5224,P001859,O001631,charge,38.49,2026-03-14 22:43:50
15431,P002189,O001926,charge,41.99,2026-03-17 18:22:24
9230,P002189,O001926,charge,41.99,2026-03-17 18:22:24


okay so i verify that it is all exact duplication so will remove it

prompt : remove the exact duplication in table payment and verify it

In [9]:
initial_count = len(tables['payments'])
tables['payments'] = tables['payments'].drop_duplicates().reset_index(drop=True)
new_count = len(tables['payments'])

print(f"Initial payments count: {initial_count:,}")
print(f"Cleaned payments count: {new_count:,}")
print(f"Removed duplicates: {initial_count - new_count}")

# Re-verify by checking if any duplicates remain
remaining_duplicates = tables['payments'].duplicated().sum()
print(f"Remaining exact duplicates: {remaining_duplicates}")

# Update the DuckDB registration
con.register('payments', tables['payments'])

Initial payments count: 16,423
Cleaned payments count: 16,393
Removed duplicates: 30
Remaining exact duplicates: 0


okay the duplication has been removed. So next i will check the consistency and formating for each table. Let start with table order

prompt : check for the data consistency and formating for table order. I expact the column order_id to be unique since it the primary key.

In [10]:
import pandas as pd

orders_df = tables['orders']

# 1. Check uniqueness of order_id
total_rows = len(orders_df)
unique_orders = orders_df['order_id'].nunique()
is_unique = total_rows == unique_orders

print(f"=== 'orders' Table Uniqueness Check ===")
print(f"Total Rows: {total_rows:,}")
print(f"Unique order_id Count: {unique_orders:,}")
print(f"Is order_id a unique Primary Key? {is_unique}\n")

# 2. Check value ranges and formats in 'orders'
print("=== 'orders' Range Check ===")
display(orders_df[['delivery_fee']].describe())
print(f"Order timestamp range: {orders_df['order_ts'].min()} to {orders_df['order_ts'].max()}")

=== 'orders' Table Uniqueness Check ===
Total Rows: 14,494
Unique order_id Count: 14,494
Is order_id a unique Primary Key? True

=== 'orders' Range Check ===


,delivery_fee
count,14494.000000
mean,2.150676
std,1.104824
min,0.000000
25%,1.490000
50%,2.490000
75%,2.990000
max,3.490000


Order timestamp range: 2026-03-01T09:13:23Z to 2026-06-30T22:58:46Z


okay everything is exactly how it expected it so will move on to check the same thing for order_lines

prompt : check for consistency and formating for table order_line espactially for the order_id whether it unique which i don't expeact it to be. Also want to check the formating of the item name and also the statistic for item value

In [12]:
import pandas as pd

order_lines = tables['order_lines']

# 1. Check uniqueness of order_id
total_rows = len(order_lines)
unique_order_ids = order_lines['order_id'].nunique()

print(f"=== 'order_lines' Grain & Key Check ===")
print(f"Total rows (line items): {total_rows:,}")
print(f"Unique orders in order_lines: {unique_order_ids:,}")
print(f"Is order_id unique in order_lines? {total_rows == unique_order_ids}")
print(f"Average items per order: {total_rows / unique_order_ids:.2f}\n")

# 2. Check formatting of item_name
print("=== 'order_lines' Item Name Formatting ===")
unique_names = order_lines['item_name'].unique()
print(f"Number of unique item names: {len(unique_names)}")
print("List of unique item names:")
print(sorted(list(unique_names)))

# 3. Calculate statistics for item values
# We'll calculate a temporary column for line value (quantity * unit_price)
order_lines['line_value'] = order_lines['quantity'] * order_lines['unit_price']

print("\n=== 'order_lines' Value Statistics ===")
display(order_lines[['quantity', 'unit_price', 'line_value']].describe())

=== 'order_lines' Grain & Key Check ===
Total rows (line items): 36,120
Unique orders in order_lines: 14,494
Is order_id unique in order_lines? False
Average items per order: 2.49

=== 'order_lines' Item Name Formatting ===
Number of unique item names: 12
List of unique item names:
['Burrito', 'Cheeseburger', 'Chicken tikka', 'Dessert', 'Falafel wrap', 'Fried rice', 'Fries', 'Margherita', 'Pad thai', 'Ramen', 'Side salad', 'Soft drink']

=== 'order_lines' Value Statistics ===


,quantity,unit_price,line_value
count,36120.000000,36120.000000,36120.000000
mean,1.995903,7.211739,14.399003
std,0.813985,3.131357,8.949223
min,1.000000,2.000000,2.000000
25%,1.000000,5.000000,7.500000
50%,2.000000,8.000000,12.000000
75%,3.000000,10.500000,21.000000
max,3.000000,12.000000,36.000000


yeah the order is not unique here which is expected and the item name formating look good as well as the statistic so will move on to table order_status_events

prompt : check for the consistency and formating for table order_status_events. I expact event id to be unique and want to check the formating of status and time

In [13]:
import pandas as pd

events_df = tables['order_status_events']

# 1. Check uniqueness of event_id
total_rows = len(events_df)
unique_events = events_df['event_id'].nunique()
is_event_unique = total_rows == unique_events

print(f"=== 'order_status_events' Key Uniqueness Check ===")
print(f"Total Rows: {total_rows:,}")
print(f"Unique event_id Count: {unique_events:,}")
print(f"Is event_id unique (Primary Key)? {is_event_unique}\n")

# 2. Check the formatting and distribution of status values
print("=== Status Formatting & Distribution ===")
status_distribution = events_df['status'].value_counts(dropna=False)
display(status_distribution)

# 3. Check the format and range of event_ts
print("\n=== Event Timestamp Check ===")
try:
    # Temporarily parse to check for invalid dates/formats
    parsed_ts = pd.to_datetime(events_df['event_ts'], errors='raise')
    print(f"Min Timestamp: {parsed_ts.min()}")
    print(f"Max Timestamp: {parsed_ts.max()}")
    print("All timestamps are in a valid date-time format.")
except Exception as e:
    print(f"Error parsing timestamps: {e}")

=== 'order_status_events' Key Uniqueness Check ===
Total Rows: 56,812
Unique event_id Count: 56,812
Is event_id unique (Primary Key)? True

=== Status Formatting & Distribution ===


,count
status,
placed,14494
accepted,14154
picked_up,13745
delivered,13549
cancelled,870



=== Event Timestamp Check ===
Min Timestamp: 2026-03-01 09:13:23+00:00
Max Timestamp: 2026-06-30 23:37:01+00:00
All timestamps are in a valid date-time format.


okay the event id is unique and is primary key which is expacted and the formating of the status and timestampt look great so will move on to table payment


prompt : check for consistency and formating for table payment. I expact the payment id to be unique. I expact the formating for payment type and payment time to be consistent and also want to check the statistic of the column amount

In [14]:
import pandas as pd

payments_df = tables['payments']

# 1. Check uniqueness of payment_id
total_rows = len(payments_df)
unique_payments = payments_df['payment_id'].nunique()
is_unique = total_rows == unique_payments

print(f"=== 'payments' Table Uniqueness Check ===")
print(f"Total Rows: {total_rows:,}")
print(f"Unique payment_id Count: {unique_payments:,}")
print(f"Is payment_id a unique Primary Key? {is_unique}\n")

# 2. Check value distribution and formatting of payment_type
print("=== Payment Type Formatting & Distribution ===")
display(payments_df['payment_type'].value_counts(dropna=False))

# 3. Check format consistency and range of paid_at time column
print("\n=== Timestamp Check (paid_at) ===")
try:
    parsed_paid_at = pd.to_datetime(payments_df['paid_at'], errors='raise')
    print(f"All paid_at values are formatted correctly as datetime.")
    print(f"Payment date range: {parsed_paid_at.min()} to {parsed_paid_at.max()}")
except Exception as e:
    print(f"Error parsing paid_at column: {e}")

# 4. Check statistics of column amount
print("\n=== 'amount' Column Statistics ===")
display(payments_df['amount'].describe())

=== 'payments' Table Uniqueness Check ===
Total Rows: 16,393
Unique payment_id Count: 16,393
Is payment_id a unique Primary Key? True

=== Payment Type Formatting & Distribution ===


,count
payment_type,
charge,14494
refund,1899



=== Timestamp Check (paid_at) ===
All paid_at values are formatted correctly as datetime.
Payment date range: 2026-03-01 09:14:35 to 2026-07-02 17:13:58

=== 'amount' Column Statistics ===


,amount
count,16393.000000
mean,30.975435
std,29.094554
min,-127.490000
25%,14.490000
50%,31.490000
75%,49.990000
max,127.490000


payment id is unique which is expacted and all the formating looks great. There are some negative value in the column amount which i want to quickly check

prompt : filter out some of the negative value in column amount in table payment to check further

In [15]:
# Filter and display a sample of negative payments
negative_payments = payments_df[payments_df['amount'] < 0]

print(f"Total negative payment records: {len(negative_payments)}")
print("\nValue distribution of payment_type for negative amounts:")
print(negative_payments['payment_type'].value_counts())

print("\nSample of negative payment records:")
display(negative_payments.head(10))

Total negative payment records: 1899

Value distribution of payment_type for negative amounts:
payment_type
refund    1899
Name: count, dtype: int64

Sample of negative payment records:


,payment_id,order_id,payment_type,amount,paid_at
6,P010253,O009054,refund,-11.99,2026-05-15 22:14:20
8,P005117,O004520,refund,-10.40,2026-04-09 03:55:12
16,P011082,O009795,refund,-4.21,2026-05-23 21:05:57
20,P000177,O000148,refund,-9.49,2026-03-04 03:48:26
26,P001778,O001559,refund,-5.26,2026-03-15 08:03:05
32,P009663,O008523,refund,-49.49,2026-05-11 10:39:27
35,P014270,O012611,refund,-9.99,2026-06-15 00:35:47
37,P008591,O007577,refund,-1.41,2026-05-05 15:27:37
59,P000055,O000044,refund,-10.43,2026-03-02 18:27:42
60,P002154,O001896,refund,-7.80,2026-03-19 08:10:04


mostly it seems to be refund which is expacted for the value to be negative but will confirm it again by unique all the payment type for every row that amount is negative

prompt : unique the payment type for all the rows that amount is negative in table payment

In [16]:
# Double check the unique payment types for all rows where amount is negative
unique_neg_types = payments_df[payments_df['amount'] < 0]['payment_type'].unique()
print("Unique payment types for negative amounts:", unique_neg_types)

Unique payment types for negative amounts: ['refund']


nice so everything looks good now so will move on to table restaurants

prompt : check for consistency and formating in table restaurant. I expact the restaurant id to be unique and for format of the rest of the column to be consistent

In [17]:
import pandas as pd

restaurants_df = tables['restaurants']

# 1. Check uniqueness of restaurant_id
total_rows = len(restaurants_df)
unique_restaurants = restaurants_df['restaurant_id'].nunique()
is_unique = total_rows == unique_restaurants

print(f"=== 'restaurants' Table Uniqueness Check ===")
print(f"Total Rows: {total_rows}")
print(f"Unique restaurant_id Count: {unique_restaurants}")
print(f"Is restaurant_id a unique Primary Key? {is_unique}\n")

# 2. Check value distribution and format of other columns
print("=== Cuisine and City Distributions ===")
print("\nCuisines:")
print(restaurants_df['cuisine'].value_counts())
print("\nCities:")
print(restaurants_df['city'].value_counts())

# 3. Check effective_from date range
print("\n=== 'effective_from' Date Range ===")
try:
    parsed_dates = pd.to_datetime(restaurants_df['effective_from'], errors='raise')
    print(f"Date Range: {parsed_dates.min()} to {parsed_dates.max()}")
except Exception as e:
    print(f"Error parsing dates: {e}")

# 4. Show sample data
print("\n=== Sample restaurant records ===")
display(restaurants_df.head(5))

=== 'restaurants' Table Uniqueness Check ===
Total Rows: 67
Unique restaurant_id Count: 64
Is restaurant_id a unique Primary Key? False

=== Cuisine and City Distributions ===

Cuisines:
cuisine
Vegan        9
Thai         8
Chinese      8
Japanese     7
Indian       7
Italian      7
Burgers      7
Mexican      6
mexican      2
thai         1
japanese     1
Burgers      1
Japanese     1
Indian       1
Italian      1
Name: count, dtype: int64

Cities:
city
London        23
Manchester    17
Birmingham    12
Bristol        7
Leeds          5
LEEDS          1
MANCHESTER     1
LONDON         1
Name: count, dtype: int64

=== 'effective_from' Date Range ===
Date Range: 2025-01-01 00:00:00 to 2026-05-27 00:00:00

=== Sample restaurant records ===


,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R064,Fresh Vegan 64,Vegan,London,2025-01-01
1,R012,Royal Burgers 12,Burgers,Bristol,2025-01-01
2,R028,Grand Burgers 28,Burgers,Birmingham,2025-01-01
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
4,R026,Corner Indian 26,Indian,London,2025-01-01


the restaurant id is not unique which is unexpected so i will investigate further and the formating of the cuisine and city is not consistent

so i will first check the restaurant id

prompt : filter out all the rows that has duplicate restaurant id in table restaurant

In [18]:
duplicates_rest = restaurants_df[restaurants_df.duplicated(subset=['restaurant_id'], keep=False)].sort_values(by='restaurant_id')
display(duplicates_rest)

,restaurant_id,restaurant_name,cuisine,city,effective_from
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
7,R007,Royal Japanese 7,Japanese,London,2026-04-14
6,R021,Grand Thai 21,Thai,LEEDS,2026-05-06
51,R021,Grand Thai 21,Thai,Manchester,2025-01-01
35,R040,Corner Vegan 40,Vegan,Bristol,2026-05-27
48,R040,Corner Vegan 40,Vegan,London,2025-01-01


it seems to be because the restaurant move so will only retain the data that is the move recent and remove the duplication

prompt : remove the duplicate restaurant id and retain on the the most recent effective_from for table restaurant and verify the remove

In [19]:
import pandas as pd

# Ensure effective_from is parsed as datetime for correct sorting
restaurants_df['effective_from'] = pd.to_datetime(restaurants_df['effective_from'])

# Sort by effective_from descending, then drop duplicates keeping the first (most recent)
cleaned_restaurants = restaurants_df.sort_values(by='effective_from', ascending=False).drop_duplicates(subset=['restaurant_id'], keep='first').reset_index(drop=True)

# Update our tables dictionary and register with DuckDB
tables['restaurants'] = cleaned_restaurants
con.register('restaurants', tables['restaurants'])

# Verify uniqueness
total_rows = len(tables['restaurants'])
unique_ids = tables['restaurants']['restaurant_id'].nunique()
print(f"=== Cleaned 'restaurants' Table Uniqueness Check ===")
print(f"Total Rows remaining: {total_rows}")
print(f"Unique restaurant_id Count: {unique_ids}")
print(f"Is restaurant_id now a unique Primary Key? {total_rows == unique_ids}\n")

# Show the cleaned records for those previously duplicated (R007, R021, R040)
display(tables['restaurants'][tables['restaurants']['restaurant_id'].isin(['R007', 'R021', 'R040'])])

=== Cleaned 'restaurants' Table Uniqueness Check ===
Total Rows remaining: 64
Unique restaurant_id Count: 64
Is restaurant_id now a unique Primary Key? True



,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R040,Corner Vegan 40,Vegan,Bristol,2026-05-27
1,R021,Grand Thai 21,Thai,LEEDS,2026-05-06
2,R007,Royal Japanese 7,Japanese,London,2026-04-14


okay i verify that it's clean now so will fix the formating of column city and cuisine. Since i don't have lot of time i will do both at once and verify it

prompt : clean the formating for column cuisine and city for table restaurant to start with capital letter and the rest can be smaller case and also remove white spact before and after the word and then print out all the value again

In [20]:
# Clean the formatting for cuisine and city columns in the restaurants table
tables['restaurants']['cuisine'] = tables['restaurants']['cuisine'].astype(str).str.strip().str.title()
tables['restaurants']['city'] = tables['restaurants']['city'].astype(str).str.strip().str.title()

# Re-register the cleaned table with DuckDB
con.register('restaurants', tables['restaurants'])

# Print out unique values and counts for verification
print("=== Cleaned Cuisine Distribution ===")
print(tables['restaurants']['cuisine'].value_counts())

print("\n=== Cleaned City Distribution ===")
print(tables['restaurants']['city'].value_counts())

=== Cleaned Cuisine Distribution ===
cuisine
Vegan       8
Thai        8
Japanese    8
Mexican     8
Chinese     8
Indian      8
Italian     8
Burgers     8
Name: count, dtype: int64

=== Cleaned City Distribution ===
city
London        23
Manchester    17
Birmingham    12
Bristol        6
Leeds          6
Name: count, dtype: int64


okay everthing look good now so will move on to the last table which is customer

prompt : check the consistency and formating for table customers. I expact the customer id to be unique and for the format for the rest of the column to be consistent

In [21]:
import pandas as pd

customers_df = tables['customers']

# 1. Check uniqueness of customer_id
total_rows = len(customers_df)
unique_customers = customers_df['customer_id'].nunique()
is_unique = total_rows == unique_customers

print("=== 'customers' Table Uniqueness Check ===")
print(f"Total Rows: {total_rows:,}")
print(f"Unique customer_id Count: {unique_customers:,}")
print(f"Is customer_id a unique Primary Key? {is_unique}\n")

# 2. Check value distribution and formatting of home_city
print("=== 'home_city' Value Distribution ===")
display(customers_df['home_city'].value_counts(dropna=False))

# 3. Check format consistency and range of signup_date
print("\n=== 'signup_date' Range Check ===")
try:
    parsed_signup_dates = pd.to_datetime(customers_df['signup_date'], errors='raise')
    print("All signup_date values are formatted correctly as datetime.")
    print(f"Signup date range: {parsed_signup_dates.min()} to {parsed_signup_dates.max()}")
except Exception as e:
    print(f"Error parsing signup_date: {e}")

# 4. Show sample data
print("\n=== Sample customer records ===")
display(customers_df.head(5))

=== 'customers' Table Uniqueness Check ===
Total Rows: 4,200
Unique customer_id Count: 4,200
Is customer_id a unique Primary Key? True

=== 'home_city' Value Distribution ===


,count
home_city,
London,1541
Manchester,857
Birmingham,697
Bristol,569
Leeds,536



=== 'signup_date' Range Check ===
All signup_date values are formatted correctly as datetime.
Signup date range: 2024-01-02 00:00:00 to 2026-03-10 00:00:00

=== Sample customer records ===


,customer_id,signup_date,home_city
0,C00001,2024-10-15,London
1,C00002,2025-05-17,Leeds
2,C00003,2026-01-25,Leeds
3,C00004,2024-09-14,London
4,C00005,2024-01-25,Manchester


okay everything looks good so will move on to build now

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

Brief : One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

so what i have to do is join the table restaurant to get cuisince and city and count the number of order from table order. for net revenue i think have to be calculate from table payment and get unit price from order_line and the delivery fee from table order. For the delivery time i think we can get from the timestamp in order_status_event



## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

Add text here as you go along. Show that you are staying in control of the build!

since im running out of time i will ask AI to prepare the data based on this

prompt : im preping table to create a report that generate from one table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

so what i have to do is join the table restaurant to get cuisince and city and count the number of order from table order. for net revenue i think have to be calculate from table payment and get unit price from order_line and the delivery fee from table order. For the delivery time i think we can get from the timestamp in order_status_event

create raw table from the table we have clean to prepare the data to aggregate for the final table

### AI: Build Clean Master Order Table
We will construct a single, comprehensive table at the order level (`orders_cleaned_augmented`).
- **Net Revenue**: Calculated directly from cash flows in `payments` (charges minus refunds for each order).
- **Delivery Time (Minutes)**: Calculated as the difference between `placed` and `delivered` timestamps for each order.

In [23]:
# Calculate net payments (charges + refunds) per order
payments_cleaned = tables['payments']
net_payments_per_order = payments_cleaned.groupby('order_id')['amount'].sum().reset_index(name='net_payment_revenue')
con.register('net_payments_per_order', net_payments_per_order)

# Calculate delivery time from order status events
events_df = tables['order_status_events']
events_df['event_ts'] = pd.to_datetime(events_df['event_ts'])

# Safely get the EARLIEST 'placed' timestamp per order
placed_events = events_df[events_df['status'] == 'placed'].groupby('order_id')['event_ts'].min().reset_index(name='placed_ts')

# Safely get the LATEST 'delivered' timestamp per order
delivered_events = events_df[events_df['status'] == 'delivered'].groupby('order_id')['event_ts'].max().reset_index(name='delivered_ts')

# Merge to calculate delivery times safely (one row per order_id)
delivery_times = pd.merge(placed_events, delivered_events, on='order_id', how='inner')
delivery_times['delivery_time_mins'] = (delivery_times['delivered_ts'] - delivery_times['placed_ts']).dt.total_seconds() / 60.0
con.register('delivery_times', delivery_times)

# Construct the augmented master orders table
orders_augmented = q("""
    SELECT
        o.order_id,
        o.customer_id,
        o.restaurant_id,
        CAST(o.order_ts AS TIMESTAMP) AS order_ts,
        o.delivery_fee,
        r.cuisine,
        r.city,
        COALESCE(p.net_payment_revenue, 0.0) AS net_revenue,
        dt.delivery_time_mins
    FROM orders o
    LEFT JOIN restaurants r ON o.restaurant_id = r.restaurant_id
    LEFT JOIN net_payments_per_order p ON o.order_id = p.order_id
    LEFT JOIN delivery_times dt ON o.order_id = dt.order_id
""")

tables['orders_augmented'] = orders_augmented
con.register('orders_augmented', orders_augmented)

print(f"=== Master Orders Table Generated ===")
print(f"Total Rows (should match original 14,494 orders): {len(orders_augmented):,}")
print(f"Duplicate order_ids in master table: {orders_augmented['order_id'].duplicated().sum()}")
display(orders_augmented.head())

=== Master Orders Table Generated ===
Total Rows (should match original 14,494 orders): 14,494
Duplicate order_ids in master table: 0


,order_id,customer_id,restaurant_id,order_ts,delivery_fee,cuisine,city,net_revenue,delivery_time_mins
0,O000001,C01161,R052,2026-03-01 09:13:23,1.49,Burgers,London,37.49,60.266667
1,O000002,C00723,R027,2026-03-01 09:15:33,2.49,Chinese,London,31.99,46.333333
2,O000003,C00781,R044,2026-03-01 09:23:42,0.00,Burgers,Birmingham,12.50,39.050000
3,O000004,C01430,R049,2026-03-01 09:28:14,1.49,Italian,Birmingham,46.99,42.450000
4,O000006,C00597,R062,2026-03-01 11:38:04,3.49,Mexican,London,68.49,36.150000


since im running out of time i will just quickly check whether the join work properly by checking for duplication and missing value

prompt : check orders_augmented for any duplication and missing value

In [25]:
# Check for duplicate order_ids in our clean master table
total_rows = len(orders_augmented)
duplicates_count = orders_augmented['order_id'].duplicated().sum()

print("=== Duplication Check ===")
print(f"Total Rows: {total_rows:,}")
print(f"Duplicate order_ids: {duplicates_count}\n")

# Profile each column for missing values
print("=== Missing Values Profile ===")
null_counts = orders_augmented.isnull().sum()
null_percentages = (null_counts / total_rows) * 100

null_profile = pd.DataFrame({
    'Missing Count': null_counts,
    'Percentage (%)': null_percentages.round(2)
})
display(null_profile)

=== Duplication Check ===
Total Rows: 14,494
Duplicate order_ids: 0

=== Missing Values Profile ===


,Missing Count,Percentage (%)
order_id,0,0.00
customer_id,0,0.00
restaurant_id,0,0.00
order_ts,0,0.00
delivery_fee,0,0.00
cuisine,0,0.00
city,0,0.00
net_revenue,0,0.00
delivery_time_mins,1085,7.49


okay will quickly move on to construct the final table

prompt : create the table based on the bried using table orders_augmented

In [24]:
# Construct the final aggregated weekly report using DuckDB
weekly_report = q("""
    SELECT
        DATE_TRUNC('week', order_ts)::DATE AS week_start_date,
        cuisine,
        city,
        COUNT(order_id) AS number_of_orders,
        ROUND(SUM(net_revenue), 2) AS net_revenue,
        ROUND(AVG(delivery_time_mins), 2) AS avg_delivery_time_mins
    FROM orders_augmented
    GROUP BY 1, 2, 3
    ORDER BY 1 ASC, 2 ASC, 3 ASC
""")

tables['weekly_report'] = weekly_report
con.register('weekly_report', weekly_report)

print("=== Final Weekly Operations Report Generated ===")
print(f"Total rows in reporting table: {len(weekly_report):,}")
display(weekly_report.head(10))

=== Final Weekly Operations Report Generated ===
Total rows in reporting table: 566


,week_start_date,cuisine,city,number_of_orders,net_revenue,avg_delivery_time_mins
0,2026-02-23,Burgers,Birmingham,4,125.99,38.81
1,2026-02-23,Burgers,Bristol,4,66.77,40.20
2,2026-02-23,Burgers,Leeds,1,0.00,NaN
3,2026-02-23,Burgers,London,11,396.42,39.68
4,2026-02-23,Chinese,Birmingham,7,117.46,34.01
5,2026-02-23,Chinese,London,8,284.93,35.07
6,2026-02-23,Chinese,Manchester,1,24.99,33.67
7,2026-02-23,Indian,Birmingham,1,43.99,42.63
8,2026-02-23,Indian,London,8,215.56,30.67
9,2026-02-23,Italian,Birmingham,8,138.94,36.24


let do a quick sanity check of the final table

prompt: check the stat for column number of order, net revenue and avg_delivery time mins

In [26]:
# Calculate descriptive statistics for key metrics in the final weekly report
print("=== Final Weekly Operations Report Summary Statistics ===")
display(weekly_report[['number_of_orders', 'net_revenue', 'avg_delivery_time_mins']].describe())

# Calculate total sum checks to reconcile with master orders table
total_orders_gold = weekly_report['number_of_orders'].sum()
total_revenue_gold = weekly_report['net_revenue'].sum()

total_orders_silver = len(tables['orders_augmented'])
total_revenue_silver = tables['orders_augmented']['net_revenue'].sum()

print("=== Reconciliation Verification Check ===")
print(f"Total Orders (Gold Layer): {total_orders_gold:,} | (Silver Layer): {total_orders_silver:,} | Match? {total_orders_gold == total_orders_silver}")
print(f"Total Net Revenue (Gold Layer): £{total_revenue_gold:,.2f} | (Silver Layer): £{total_revenue_silver:,.2f} | Match? {round(total_revenue_gold, 2) == round(total_revenue_silver, 2)}")

=== Final Weekly Operations Report Summary Statistics ===


,number_of_orders,net_revenue,avg_delivery_time_mins
count,566.000000,566.000000,564.000000
mean,25.607774,897.138357,36.156011
std,20.304362,731.470829,4.280677
min,1.000000,0.000000,22.720000
25%,12.000000,408.077500,34.467500
50%,20.000000,705.700000,35.790000
75%,32.000000,1140.642500,37.110000
max,103.000000,3816.860000,86.230000


=== Reconciliation Verification Check ===
Total Orders (Gold Layer): 14,494 | (Silver Layer): 14,494 | Match? True
Total Net Revenue (Gold Layer): £507,780.31 | (Silver Layer): £507,780.31 | Match? True


okay the AI also di the reconciliation verification check which prove that the data from silver and gold match

## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Add text here as you go along. Show that you are staying in control of the build!

checks have been added throughout the process

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

prompt already been added throughout the process

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*



*   the process of preparing table to be build into the final table is not check throughly



## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.